# 01_parse_recipes

Parse recipes from cookbooks and store each cookbook into a json file.

## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = '67okpxpxehbtif'
os.environ['DataZoneDomainId'] = 'dzd-dlubrycszru95z'
os.environ['DataZoneEnvironmentId'] = 'c5n6kyh3ks4s6f'
os.environ['DataZoneDomainRegion'] = 'us-east-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "67okpxpxehbtif",
                "DataZoneDomainId": "dzd-dlubrycszru95z",
                "DataZoneEnvironmentId": "c5n6kyh3ks4s6f",
                "DataZoneDomainRegion": "us-east-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

**1. Notebook Setup**

In [0]:
#import statements
import sagemaker
import os
import xml.etree.ElementTree as ET
import string
import json
import boto3
import re
import unicodedata
import html
import codecs

In [0]:
# Use the correct bucket name without 'sagemaker-' prefix
bucket = "feeding-america-historic-cookbooks"

sess = sagemaker.Session(default_bucket=bucket)

print(f"Using bucket: {bucket}")


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


Using bucket: feeding-america-historic-cookbooks


**2. Define classes to parse cookbooks**

In [0]:
class RecipeParser:
    """Parses XML recipe elements into structured dictionaries."""

    def __init__(self, recipe, recipe_id):
        self.recipe = recipe
        self.recipe_id = recipe_id
        self.recipe_dict = {}

    def get_recipe_title(self):
        purpose = self.recipe.find('p/purpose')

        if purpose is None:
            return None

        title = "".join(purpose.itertext()).strip()

        if title.endswith('.'):
            title = title[:-1]

        return title

    def get_ingredients(self):
        ingredients = self.recipe.findall(".//ingredient")
        cleaned_ingredients = []

        for i in ingredients:
            if i.text:
                text = i.text.lower().strip(string.punctuation)
                cleaned_ingredients.append(text)

        return list(dict.fromkeys(cleaned_ingredients))

    def get_cooking_instructions(self):
        instructions = "".join(self.recipe.itertext()).strip()

        instructions = ' '.join(instructions.split())
        
        return instructions 
        # instructions = "".join(self.recipe.itertext()).strip()
        # # Splitting and taking index 1 based on your specific logic
        # lines = instructions.split("\n")
        # return lines[1].strip() if len(lines) > 1 else instructions

    def get_recipe_category(self):
        return self.recipe.attrib.get('class1')

    def recipe_to_dict(self):
        ingredients = self.get_ingredients()
        instructions = self.get_cooking_instructions()
        title = self.get_recipe_title()
        category = self.get_recipe_category()

        if (ingredients and len(ingredients) >= 2) and (instructions and len(instructions) >= 30) and title:
            self.recipe_dict['recipe_id'] = self.recipe_id
            self.recipe_dict['recipe_title'] = title
            self.recipe_dict['recipe_category'] = category
            self.recipe_dict['recipe_ingredients'] = ingredients
            self.recipe_dict['recipe_instructions'] = instructions

            return self.recipe_dict

        return None

In [0]:
class BookParser:
    """Parses an entire XML recipe book into a dictionary structure."""

    def __init__(self, s3_client, bucket, s3_key):
        self.s3 = s3_client
        self.bucket = bucket
        self.s3_key = s3_key

        obj = self.s3.get_object(Bucket=bucket, Key=s3_key)
        xml_content = obj["Body"].read()

        self.tree = ET.fromstring(xml_content)
        self.root = self.tree
        self.book_dict = {}

    def get_book_attr(self):
        attribs = self.root.attrib

        return {
            'book_id': attribs.get('bookID'),
            'book_type': attribs.get('type'),
            'book_class': attribs.get('class1'),
            'book_region': attribs.get('region'),
            'book_ethnic_group': attribs.get('ethnicgroup'),
            'book_historic_period': attribs.get('histperiod')
        }

    def get_meta_attr(self):
        meta_attr = {}
        meta_element = self.root.find('meta')

        if meta_element is not None:
            meta_attr['book_title'] = self.root.findtext('meta/dcTitle')
            meta_attr['book_creator'] = self.root.findtext('meta/dcCreator')
            meta_attr['book_description'] = self.root.findtext('meta/dcDescription')
            meta_attr['book_publisher'] = self.root.findtext('meta/dcPublisher')
            meta_attr['book_year'] = self.root.findtext('meta/dcDate')

        return meta_attr

    def book_to_dict(self, limit=None):
        book_attr = self.get_book_attr()

        self.book_dict = {
            'attributes': book_attr,
            'metadata': self.get_meta_attr(),
            'recipes': {}
        }

        recipe_elements = self.root.findall(".//recipe")

        count = 0

        for recipe_obj in recipe_elements:
            count += 1

            if limit is not None and count > limit:
                break

            recipe_id = f"{book_attr['book_id']}_{count}"

            parser = RecipeParser(recipe_obj, recipe_id)
            recipe_dict = parser.recipe_to_dict()

            if recipe_dict is not None:
                self.book_dict['recipes'][recipe_id] = recipe_dict

        self.book_dict['recipe_num'] = len(self.book_dict['recipes'])

        return self.book_dict




**3. Parse cookbooks and save to json file**

In [0]:

s3 = boto3.client('s3')

input_folder = 'cookbook_textencoded/'
output_folder = 'processed_cookbooks/'

paginator = s3.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=bucket, Prefix=input_folder)

book_count = 0

for page in pages:
    if 'Contents' not in page:
        continue

    for obj in page['Contents']:
        source_key = obj['Key']

        if source_key == input_folder:
            continue

        file_name = os.path.basename(source_key)

        if not file_name.endswith('.xml'):
            continue

        book_count += 1
        if book_count > 5:
            break

        print(f"Processing: {source_key}")

        book_name = file_name.split('.')[0]

        try:
            book = BookParser(s3_client=s3, bucket=bucket, s3_key=source_key)
            book_dict = book.book_to_dict()

            if book_dict["recipe_num"] == 0:
                print(f'  - Recipes found: {book_dict["recipe_num"]}')
                print('    No Json file will be created. \n')
                

            else:

                json_content = json.dumps(book_dict, indent=4, ensure_ascii=False)

                json_destination_key = f'{output_folder}{book_name}.json'

                s3.put_object(
                    Bucket=bucket,
                    Key=json_destination_key,
                    Body=json_content,
                    ContentType='application/json'
                )

                print(f'  - Recipes found: {book_dict["recipe_num"]}\n')
                

        except Exception as e:
            print(f'✗ Failed on {source_key}: {e}')

    if book_count > 5:
        break


Processing: cookbook_textencoded/amem.xml
  - Recipes found: 0
    No Json file will be created. 

Processing: cookbook_textencoded/amwh.xml


  - Recipes found: 3

Processing: cookbook_textencoded/army.xml
  - Recipes found: 369

Processing: cookbook_textencoded/aunt.xml


  - Recipes found: 910

Processing: cookbook_textencoded/bart.xml
  - Recipes found: 171



## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()